# Approach 2: MobileNetV2 transfer learning

Train a new classifier head on ImageNet-pretrained MobileNetV2. The feature backbone is frozen, training augmentation is applied only to the training split, and early stopping monitors validation loss. The held-out test set is not used here.

In [ ]:
from pathlib import Path
import copy
import sys

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.dataset import get_dataset_splits
from src.torch_dataset import OrangeImageDataset
from src.dl_model import build_orange_classifier

splits = get_dataset_splits(PROJECT_ROOT / 'data' / 'raw', validation_size=0.15, random_state=42)
class_names = splits['class_names']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '| Classes:', class_names)
print({name: len(splits[name]) for name in ('train', 'validation', 'test')})

In [ ]:
image_size = 224
imagenet_mean = (0.485, 0.456, 0.406)
imagenet_std = (0.229, 0.224, 0.225)
train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])
evaluation_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])
train_dataset = OrangeImageDataset(splits['train'], class_names, train_transform)
validation_dataset = OrangeImageDataset(splits['validation'], class_names, evaluation_transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
validation_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False, num_workers=0)

train_labels = [class_names.index(label) for _, label in splits['train']]
class_counts = np.bincount(train_labels, minlength=len(class_names))
class_weights = len(train_labels) / (len(class_names) * class_counts)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))
model = build_orange_classifier(num_classes=len(class_names), pretrained=True).to(device)
optimizer = torch.optim.Adam((p for p in model.parameters() if p.requires_grad), lr=1e-3)
print('Class weights:', dict(zip(class_names, class_weights.round(3))))

In [ ]:
max_epochs = 12
patience = 3
best_validation_loss = float('inf')
best_state = None
epochs_without_improvement = 0
history = []

for epoch in range(max_epochs):
    model.train()
    train_loss_total = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        train_loss_total += loss.item() * labels.size(0)

    model.eval()
    validation_loss_total = 0.0
    validation_correct = 0
    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            validation_loss_total += criterion(logits, labels).item() * labels.size(0)
            validation_correct += (logits.argmax(dim=1) == labels).sum().item()

    train_loss = train_loss_total / len(train_dataset)
    validation_loss = validation_loss_total / len(validation_dataset)
    validation_accuracy = validation_correct / len(validation_dataset)
    history.append({'epoch': epoch + 1, 'train_loss': train_loss, 'validation_loss': validation_loss, 'validation_accuracy': validation_accuracy})
    print('Epoch {}/{}: train loss {:.4f}, validation loss {:.4f}, validation accuracy {:.3f}'.format(epoch + 1, max_epochs, train_loss, validation_loss, validation_accuracy))

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print('Early stopping.')
            break

if best_state is None:
    raise RuntimeError('Training did not produce a best model checkpoint.')
model.load_state_dict(best_state)
model_dir = PROJECT_ROOT / 'models'
model_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = model_dir / 'orange_mobilenet_v2.pth'
torch.save({'model_state_dict': model.state_dict(), 'class_names': class_names, 'image_size': image_size}, checkpoint_path)
print('Saved:', checkpoint_path)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
history_frame = pd.DataFrame(history)
display(history_frame)
history_frame.plot(x='epoch', y=['train_loss', 'validation_loss'], marker='o', ylabel='Cross-entropy loss', title='MobileNetV2 training and validation loss')
figure_dir = PROJECT_ROOT / 'reports' / 'figures'
figure_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(figure_dir / 'training_curves.png', dpi=160, bbox_inches='tight')
plt.show()